# Notebook for Library merging

In [1]:
# Imports
from itertools import product
from sklearn.model_selection import ParameterGrid
from Models.config_parser import load_config, load_dataset_config
from Models.dataset_and_models import Dataset, Model

In [6]:
# Load configurations
CONFIG = "../configs/config-test.yaml"

model_config   = load_config(CONFIG)        # {model_key: {param: [values]}}
dataset_config = load_dataset_config(CONFIG) # {dataset_key: {n_features: [...], n_samples: [...]}}

In [7]:
# All model × hyperparameter combinations
model_runs = [
    (model_key, params)
    for model_key, param_grid in model_config.items()
    for params in ParameterGrid(param_grid)
]

# All dataset × (n_features, n_samples) combinations
dataset_runs = [
    (dataset_key, params)
    for dataset_key, param_grid in dataset_config.items()
    for params in ParameterGrid(param_grid)
]


In [8]:
# Full cross-product: one entry per benchmark run
benchmark_runs = list(product(model_runs, dataset_runs))
print(f"{len(model_runs)} model configs × {len(dataset_runs)} dataset configs = {len(benchmark_runs)} total runs")

# Load a single dataset variant
dataset_key, dataset_params = dataset_runs[0]
ds = Dataset[dataset_key.upper()].load_dataset(**dataset_params)
X, y = ds["X"], ds["y"]

1 model configs × 5 dataset configs = 5 total runs


### Iterate trough every combination and train the model for explanation

In [9]:
from tqdm import tqdm
from Benchmarking import BenchmarkRunner
from Benchmarking.backends import ShapTrueValueBackend, ShapIQTrueValueBackend

runner = BenchmarkRunner(
    backends=[ShapTrueValueBackend, ShapIQTrueValueBackend],
    output_csv="../Benchmarking/results.csv",
    n_background=100,
    n_eval=None,
)

# Only tree models are compatible with ShapTrueValueBackend (TreeExplainer)
tree_model_keys = ["random_forest", "decision_tree", "gradient_boosting"]

for dataset_key, dataset_params in tqdm(dataset_runs, desc="datasets"):
    dataset_enum = Dataset[dataset_key.upper()]
    ds = dataset_enum.load_dataset(**dataset_params)
    X, y = ds["X"], ds["y"]

    tree_runs = [(mk, mp) for mk, mp in model_runs if mk in tree_model_keys]

    for model_key, model_params in tqdm(tree_runs, leave=False, desc="models"):
        trainer = Model[model_key.upper()].get_model_with_params(dataset_enum, model_params)
        trainer.fit(X, y, task=ds["task"])

        runner.run(
            model=trainer.get_model(),
            X=X,
            run_meta={
                "dataset": dataset_key,
                "model": model_key,
                "n_features": dataset_params.get("n_features"),
                "n_samples": dataset_params.get("n_samples"),
            },
        )

print("Done. Results written to Benchmarking/results.csv")

datasets: 100%|██████████| 5/5 [02:58<00:00, 35.67s/it]

Done. Results written to Benchmarking/results.csv


In [10]:
import pandas as pd

results = pd.read_csv("../Benchmarking/results.csv")
print(f"{len(results)} rows written")
results.head(10)

14 rows written


,dataset,model,n_features,n_samples,backend,library,computation_type,n_eval,runtime_s,mean_abs_diff,sign_agreement,mean_sample_rho,reference_backend
0,california_housing,random_forest,1,2000,shap_true_value,shap,true_value,1900,0.0083,NaN,NaN,NaN,NaN
1,california_housing,random_forest,1,2000,shapiq_true_value,shapiq,true_value,1900,74.3227,0.018114,0.899474,NaN,shap_true_value
2,california_housing,decision_tree,1,2000,shap_true_value,shap,true_value,1900,0.0024,NaN,NaN,NaN,NaN
3,california_housing,decision_tree,1,2000,shapiq_true_value,shapiq,true_value,1900,38.6304,0.014087,1.000000,NaN,shap_true_value
4,california_housing,random_forest,1,1000,shap_true_value,shap,true_value,900,0.0057,NaN,NaN,NaN,NaN
5,california_housing,random_forest,1,1000,shapiq_true_value,shapiq,true_value,900,34.9691,0.016017,0.995556,NaN,shap_true_value
6,california_housing,random_forest,2,1000,shap_true_value,shap,true_value,900,0.0046,NaN,NaN,NaN,NaN
7,california_housing,random_forest,2,1000,shapiq_true_value,shapiq,true_value,900,35.6728,0.053426,0.797222,0.640000,shap_true_value
8,california_housing,random_forest,3,1000,shap_true_value,shap,true_value,900,0.0042,NaN,NaN,NaN,NaN
9,california_housing,random_forest,3,1000,shapiq_true_value,shapiq,true_value,900,35.6678,0.031100,0.839630,0.720556,shap_true_value
